# COMP5329 — Deep Learning
**Tutorial 8 — The SSM duality: recurrent ↔ convolutional**

**Semester 1, 2026**

---

### Learning Objectives

By the end of this tutorial you will be able to:

1. Explain why standard softmax attention has $O(T^2)$ complexity, and why this becomes a bottleneck as context windows grow.
2. Derive the closed-form unroll of a linear SSM recurrence and recognise it as a 1-D causal convolution.
3. Implement both the recurrent form (O(1)-state streaming inference) and the convolutional form (parallel training) of the same SSM and verify they produce identical outputs.
4. Explain what SSMs give up in exchange for this duality, and what Mamba (S6) gives up on top of that to recover input-dependent gating.

### Topic Coverage

Week 8 covers **Transformer variants and efficient sequence models**. The full topic list (see `Week8_Self_Study.ipynb`) is:

- 📖 **Quick review** — RNN, self-attention, Transformer recap *(self-study)*
- 📖 **BERT** — bidirectional encoder, masked language modelling, fine-tuning *(self-study)*
- 📖 **GPT** — autoregressive decoder, causal masking, scaling laws *(self-study)*
- 📖 **Vision Transformer (ViT)** — patchification, positional encoding for images, inductive-bias trade-off *(self-study; Exam Q3 revisits the inductive-bias argument)*
- ✅ **Efficient sequence modelling — SSMs** — continuous-to-discrete derivation, recurrent ↔ convolutional duality, from-scratch implementation of both forms *(tutorial)*
- 📖 **Mamba (S6)** — input-dependent matrices, parallel scan, selective state updates *(self-study; Exam Q2 references this)*
- 📖 **Grand comparison** — RNN vs. Transformer vs. SSM vs. Mamba across FLOPs, memory, parallelism *(self-study)*

Due to time constraints, the live tutorial focuses on the **intellectually densest block — the SSM duality** — because it is the single idea that connects Week 7's RNN view with a convolutional training path. BERT, GPT, ViT, and the deeper Mamba material are left as self-study; the self-study notebook works through them with full derivations and code.

The live session is organised into three parts: **Part A** — tutor walkthrough, **Part B** — in-class coding exercise, **Part C** — exam-style Q&A.

---
# Part A · Tutor Walkthrough

## 1. From continuous SSM to a linear RNN

A **state space model** (Kalman, 1960) describes how an internal state $h(t)$ evolves under a continuous input $x(t)$ and how it is observed as an output $y(t)$:

$$h'(t) = A\, h(t) + B\, x(t), \qquad y(t) = C\, h(t)$$

For discrete sequences (tokens, samples), we discretise with **first-order Euler** at step size $\Delta$:

$$h_t \approx h_{t-1} + \Delta\,\big(A\, h_{t-1} + B\, x_t\big) = (I + \Delta A)\, h_{t-1} + \Delta B\, x_t$$

Absorbing constants, let $\bar A = I + \Delta A$ and $\bar B = \Delta B$. The discrete SSM is:

$$\boxed{\,h_t = \bar A\, h_{t-1} + \bar B\, x_t, \qquad y_t = C\, h_t\,}$$

**This is a linear RNN.** Same shape as a Vanilla RNN from Week 7, *minus* the $\tanh$ non-linearity. That missing non-linearity is the load-bearing simplification that everything else in this tutorial rests on — keep it in mind.

**The question this tutorial answers:**
> *Can we compute the whole output $y_{1:T}$ in parallel (without running the recurrence one step at a time) while still preserving the O(1)-state streaming inference an RNN gives us?*

Spoiler: yes. The trick is to unroll the recurrence in closed form and recognise the result as a **1-D causal convolution**. You will derive this on paper, then implement both forms and prove they are numerically identical.

---
# Part B · In-Class Exercise

## 2. Implement the SSM in two forms

You will write **two functions** that both compute $y_{1:T}$ from the same input $x_{1:T}$ and the same matrices $\bar A,\ \bar B,\ C$:

- `ssm_recurrent` — steps through the recurrence token-by-token (the "RNN view").
- `ssm_convolutional` — builds a kernel $K$ of length $T$ and computes $y = K * x$ as a single 1-D causal convolution (the "parallel training view").

When both are right, the `assert` at the bottom turns green: the two forms are numerically identical on a random test input.

In [ ]:
# Setup (read-only — do not modify)
import torch
import torch.nn.functional as F

torch.manual_seed(0)
T, d = 20, 4

# A small stable SSM. Spectral radius of A_bar < 1 to avoid blow-up over T=20 steps.
A_bar = 0.9 * torch.eye(d) + 0.01 * torch.randn(d, d)
B_bar = torch.randn(d, 1)        # (d, 1) — one scalar input channel
C_bar = torch.randn(1, d)        # (1, d) — one scalar output channel
x     = torch.randn(T)           # scalar input sequence

print(f"A_bar shape: {tuple(A_bar.shape)}, spectral radius ≈ {torch.linalg.eigvals(A_bar).abs().max().item():.3f}")
print(f"B_bar shape: {tuple(B_bar.shape)}, C_bar shape: {tuple(C_bar.shape)}")
print(f"x shape: {tuple(x.shape)}")


### Task A · Recurrent form

Step through the recurrence one token at a time. State $h$ has shape `(d,)`; update it in place.

$$h_t = \bar A\, h_{t-1} + \bar B\, x_t, \qquad y_t = C\, h_t$$


In [ ]:
def ssm_recurrent(A_bar, B_bar, C_bar, x):
    """Linear SSM in recurrent form. O(d) state per step — RNN-style."""
    T, d = x.shape[0], A_bar.shape[0]
    h = torch.zeros(d)
    y = torch.zeros(T)
    b = B_bar.squeeze(-1)                      # (d,), one scalar input channel
    for t in range(T):
        # TODO (2 lines):
        # 1. update h ← A_bar @ h + b * x[t]
        # 2. y[t] = (C_bar @ h).squeeze()
        pass
    return y


### Task B · Convolutional form

**Derivation you should do on paper first** (this is on the lecture slides).
Unroll the recurrence with $h_0 = 0$:

$$h_1 = \bar B\, x_1, \quad h_2 = \bar A \bar B\, x_1 + \bar B\, x_2, \quad h_3 = \bar A^2 \bar B\, x_1 + \bar A \bar B\, x_2 + \bar B\, x_3, \quad \ldots$$

In closed form:

$$h_t = \sum_{i=0}^{t} \bar A^{\,t-i}\, \bar B\, x_i \ \Longrightarrow\ y_t = \sum_{i=0}^{t} \underbrace{\big(C\, \bar A^{\,t-i}\, \bar B\big)}_{K_{t-i}}\, x_i$$

So if you define the length-$T$ kernel $K$ with entries $K_i = C\, \bar A^{\,i}\, \bar B$, then $y_t = (K * x)_t$ — a single 1-D causal convolution.

**Two efficiency notes:**

1. Compute $K$ **incrementally**: maintain a running tensor `A_pow_B` of shape `(d, 1)`, starting at $\bar B$, and left-multiply by $\bar A$ each step. Don't call `torch.matrix_power(A_bar, i)` inside a loop — that is quadratic work on a linear problem.
2. For the convolution itself, a double loop is perfectly fine for $T=20$. A faster alternative (shown in the tutor walkthrough) is `F.conv1d` with the kernel **flipped** and the input left-padded by $T-1$.


In [ ]:
def ssm_convolutional(A_bar, B_bar, C_bar, x):
    """Linear SSM as a single 1-D causal convolution."""
    T = x.shape[0]
    d = A_bar.shape[0]

    # TODO 1: build the length-T kernel K incrementally.
    #         K[i] should equal C_bar @ (A_bar ** i) @ B_bar  (scalar).
    #         Maintain A_pow_B of shape (d, 1), starting at B_bar, and left-multiply by A_bar each step.
    K = torch.zeros(T)
    # ... your code here ...

    # TODO 2: causal convolution y[t] = sum_{i<=t} K[t-i] * x[i]
    #         A double loop is fine. Faster: F.conv1d with a flipped kernel + left-pad by T-1.
    y = torch.zeros(T)
    # ... your code here ...

    return y


In [ ]:
# Auto-verify — turns green once both Task A and Task B are correct.
y_rec  = ssm_recurrent(A_bar, B_bar, C_bar, x)
y_conv = ssm_convolutional(A_bar, B_bar, C_bar, x)

print(f"y_rec[:5]  = {y_rec[:5].tolist()}")
print(f"y_conv[:5] = {y_conv[:5].tolist()}")

assert torch.allclose(y_rec, y_conv, atol=1e-4), "recurrent form != convolutional form"
print()
print("✓ recurrent and convolutional forms agree on T=20")


### Bonus · Three reflection prompts (for fast finishers)

Don't start these until the main `assert` passes. They are deliberately open-ended — write your answers in the comments of the cell below (or discuss with a neighbour).


In [ ]:
# Bonus 1 — stability
# Try replacing A_bar with `1.05 * torch.eye(d)` (spectral radius > 1).
# Re-run ssm_recurrent and ssm_convolutional. Which one explodes first?
# Why does this motivate HiPPO initialization? (The self-study notebook has the full answer.)

# Bonus 2 — long sequences
# Set T = 10_000 and re-run both forms. Note the peak memory of each:
#   - recurrent: only keeps h of shape (d,) → O(d)
#   - convolutional: materialises the length-T kernel K → O(T)
# For T = 10^6 on a 7B-parameter SSM, which form do you want at inference time?

# Bonus 3 — what Mamba gives up
# Suppose A_bar, B_bar, C_bar become functions of x[t] (so each timestep has its own matrices).
# Can you still write K[i] = C @ A^i @ B and get a single convolution? Why not?
# This is exactly why Mamba abandons the convolution form and uses a parallel scan instead.


---
# Part C · Exam-Style Questions

Attempt each question on paper first. The answer sketch under each question is collapsed — expand only after you've written your own attempt.

---

### Q1 · Why a linear SSM is a convolution, and softmax attention is not

The SSM recurrence is
$$h_t = \bar A\, h_{t-1} + \bar B\, x_t, \qquad y_t = C\, h_t$$

**(a)** Starting from this recurrence (assume $h_0 = 0$), derive a length-$T$ sequence $K$ such that $y = K * x$ (causal convolution). Give the explicit expression for $K_i$.

**(b)** Standard scaled dot-product attention computes $y_t = \sum_{i\le t} \alpha_{t,i}\, v_i$ with $\alpha_{t,i} = \exp(q_t^\top k_i)/Z_t$. Explain why this **cannot** be written as $y = K * x$ for any $K$ that is independent of the input. Point to the specific algebraic obstruction.

*Your answer:*

<details><summary><strong>Answer sketch — Q1</strong> (open after attempting)</summary>

**(a)** Unroll the recurrence with $h_0 = 0$:
$$h_t = \sum_{i=0}^{t} \bar A^{\,t-i}\, \bar B\, x_i, \qquad y_t = \sum_{i=0}^{t} \big(C\,\bar A^{\,t-i}\,\bar B\big)\, x_i$$
Define $K_i = C\,\bar A^{\,i}\,\bar B$. Then $y_t = \sum_{i=0}^{t} K_{t-i}\, x_i = (K * x)_t$. The algebraic property that makes this possible is that $\bar A, \bar B, C$ are **constants** (independent of $t$ and of $x$), so the weight linking $x_i$ to $y_t$ depends only on the *gap* $t - i$ — precisely the definition of a convolution kernel.

**(b)** The attention weight $\alpha_{t,i}$ depends on both $t$ (through $q_t$) and $i$ (through $k_i$) through an inner product $q_t^\top k_i$ — the two indices are **entangled, not separable**. So the weight is not a function of $t-i$ alone. On top of that, $Z_t = \sum_{j\le t}\exp(q_t^\top k_j)$ is a softmax normaliser that depends on the entire prefix and is non-linear — even the numerator $\exp(q_t^\top k_i)v_i$ would not separate. No fixed, input-independent $K$ can reproduce these weights, so no convolution form exists.

</details>


### Q2 · Is an SSM just a linear RNN?

Compare Week 7's Vanilla RNN with Week 8's SSM:

$$\text{RNN: } h_t = \tanh(W_h h_{t-1} + W_x x_t), \qquad \text{SSM: } h_t = \bar A\, h_{t-1} + \bar B\, x_t$$

**(a)** Identify **two** essential differences between these two updates.

**(b)** LSTM and Vanilla RNN training is inherently sequential along time, while SSM training can be done in parallel. Using your answer to (a), explain the **precise restriction** that SSM accepts in exchange for parallel training. Then: Mamba (S6) makes $\bar A, \bar B, C$ depend on the input $x_t$ — what does it lose, and why must it introduce a new algorithmic trick (parallel scan) to recover parallel training?

*Your answer:*



<details><summary><strong>Answer sketch — Q2</strong> (open after attempting)</summary>

**(a)** Two differences:

1. **Non-linearity.** The RNN applies $\tanh$ at every step; the SSM is a purely linear recurrence. This is the load-bearing difference: strip the $\tanh$ out of a Vanilla RNN and you get *exactly* an SSM with $\bar A = W_h,\ \bar B = W_x$ (no output projection).
2. **Explicit state/output separation.** The SSM cleanly factors the model into a latent state update ($h_t = \bar A h_{t-1} + \bar B x_t$) and an output projection ($y_t = C h_t$) with a distinct $C$ matrix. This lets the latent state dimension be chosen independently of the observable input/output dimension — the "state expansion" that SSM literature (and Mamba's hardware-aware design) leans on. Vanilla RNN usually conflates state with output or tacks on a trivial linear layer, making state expansion awkward.

**(b)** Because the SSM is a **linear recurrence with constant coefficients**, it has the closed form $h_t = \sum_{i\le t}\bar A^{\,t-i}\bar B x_i$, which equals a single 1-D convolution — fully parallelisable on a GPU in $O(n \log n)$ with FFT or $O(nT)$ naively. The Vanilla RNN's $\tanh$ destroys linearity; even without $\tanh$, an LSTM's gates $f_t, i_t$ depend on $h_{t-1}$, which depends on $f_{t-1}$, and so on — no closed form exists. **The precise restriction SSM accepts is: the state update must be linear AND the matrices must be input-independent.** Mamba gives up the second half of that restriction (matrices become functions of $x_t$), which means $K_i = C\bar A^i\bar B$ is no longer meaningful — every timestep has its own $\bar A$, so the weight linking $x_i$ to $y_t$ is no longer a function of $t-i$ alone, and the convolution form disappears. Mamba recovers parallel training via a **parallel scan** (associative scan on the per-step linear operators), which trades the clean kernel $K$ for a log-depth tree of pairwise combinations.

</details>


### Q3 · ViT's patch-size trade-off and the cost of losing CNN's inductive bias

A Vision Transformer (lecture slides 32–41) splits a $224\times 224$ image into non-overlapping $P\times P$ patches, linearly projects each patch into a token, adds a positional encoding, and feeds the sequence into a standard Transformer encoder.

**(a)** Suppose we change the patch size from $P=16$ to $P=32$. Compute explicitly how (i) the sequence length $T$, (ii) the FLOPs of a single self-attention layer (which scale as $T^2 d$), and (iii) the spatial area each token covers change. Use concrete numbers.

**(b)** ViT underperforms a strong CNN (e.g. EfficientNet-B7) when both are trained only on ImageNet-1K, but surpasses it when pre-trained on JFT-300M. Using Week 4's concept of **inductive bias**, explain what built-in assumptions CNNs make that ViT does not, and why those assumptions help on small data but become a ceiling on large data.

*Your answer:*



<details><summary><strong>Answer sketch — Q3</strong> (open after attempting)</summary>

**(a)** With $P=16$: $T = (224/16)^2 = 196$. With $P=32$: $T = (224/32)^2 = 49$. So (i) $T$ drops from 196 to 49 (**factor ~4×**); (ii) self-attention FLOPs scale as $T^2 d$, so they drop by $(196/49)^2 = 16$× (**16× cheaper**); (iii) the area covered by a single token goes from $16\times16 = 256$ pixels to $32\times32 = 1024$ pixels (**4× coarser**). Trade-off: larger patches = cheaper attention + coarser spatial granularity; smaller patches = more expressive + quadratically more expensive.

**(b)** Convolutional layers bake in three inductive biases: **locality** (a filter only looks at a small neighbourhood), **translation equivariance** (the same filter slides across the whole image), and **hierarchical composition** (shallow layers learn textures, deep layers learn semantics). These are hand-engineered priors matched to natural-image statistics. ViT's self-attention has none of these — it is a global, permutation-respecting pairwise operator, and any spatial structure must be learned from data. On **small** datasets (ImageNet-1K ≈ 1.2M), CNN's priors are exactly right; ViT cannot see enough images to rediscover them and underperforms. On **large** datasets (JFT-300M ≈ 300M), ViT has enough data to learn more flexible spatial organisations than the hard-coded CNN priors allow — including global, long-range dependencies in a single layer — and the rigid CNN priors become a ceiling. In one line: **inductive bias is a double-edged sword — it does your homework for you when data is scarce, and holds you back when data is abundant.**

</details>
